# 03 Search C: where each provider's API is offered

**Purpose.** This notebook turns our hand-made Search C table into `provider_availability.csv`, one row per provider, product and place (Hong Kong, mainland China, Singapore, Taiwan). It is the evidence behind the words "unavailable or restricted" in Q1.

**Question served.** Q1 (availability side), Q5.

**This dataset is not downloaded.** It is our own table, built on 16 October 2026 from each provider's official country-list or terms page. The rows are typed in this notebook so that every value and link is visible.

**How to run.** Run all cells from top to bottom. No internet is needed. An optional cell (off by default) checks that the links still open.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Hand-built table, typed into this notebook. Nothing is downloaded |
| Where it comes from | Seven official country-list or terms pages, with links in Section 1 |
| Licence and authorisation | Public provider pages, read as an ordinary visitor with no login. We record place names and our own wording, and cite the provider. Details in Section 1 |
| Expected time | notebook under 2 seconds (measured 1.3 seconds). Optional link check 10 to 30 seconds |
| Data as of | pages read on 2026-10-16. Page dates are in `page_date` where the page shows one |
| To update | Re-read the pages by hand. See the update section below |

## 1. Sources, reference links and authorisation

All pages were opened on **2026-10-16**.

| # | Provider and product | Page | Page date |
|---|---|---|---|
| 1 | OpenAI API | https://developers.openai.com/api/docs/supported-countries (redirected from https://platform.openai.com/docs/supported-countries) | none shown |
| 2 | Anthropic API and Claude | https://www.anthropic.com/supported-countries | none shown (footer copyright 2026) |
| 3 | Google Gemini API | https://ai.google.dev/gemini-api/docs/available-regions | last updated 2026-04-28 UTC |
| 4 | DeepSeek Open Platform API | https://cdn.deepseek.com/policies/en-UK/deepseek-open-platform-terms-of-service.html and https://api-docs.deepseek.com/ | effective 2026-04-29 |
| 5 | Alibaba Cloud Model Studio | https://www.alibabacloud.com/help/en/model-studio/regions.md | none shown |
| 6 | Z.ai API (international) | https://docs.z.ai/legal-agreement/terms-of-use.md and https://docs.z.ai/guides/overview/overview | terms dated 2026-04-14 |
| 7 | Kimi open platform (international) | https://platform.kimi.ai/docs/agreement/modeluse | last updated 2026-07-30 |

**Authorisation and reuse.** These are public provider pages, read as an ordinary visitor with no login, scraping or workaround. We record what each page says in our own words and keep only the place name as the quote. The pages remain the providers' copyright. Cite the provider and link the page. This is a research summary and not legal advice.

**What the table does not show.** The pages say where service is *offered*. They do not say how a provider enforces it (for example by IP address or payment country). Test access on your own account before any claim. We give no advice on getting around any restriction.

## 2. Table structure and coding scheme

### 2.1 Output structure: `processed/provider_availability.csv`

One row per provider, product and place (7 products times 4 places gives 28 rows).

| Column | Type | Meaning |
|---|---|---|
| `provider` | text | OpenAI, Anthropic, Google, DeepSeek, Alibaba Cloud, Z.ai, Moonshot AI (Kimi) |
| `product` | text | API, Gemini API, Open Platform API, Model Studio and so on |
| `site` | text | international (all rows in this version) |
| `place` | text | Hong Kong, Mainland China, Singapore, Taiwan |
| `status` | text | supported, not_listed, excluded_stated, no_list_found |
| `page_date` | date or empty | last-updated date shown on the page, empty when none is shown |
| `date_viewed` | date | 2026-10-16 |
| `source_url` | text | the page the row was read from |
| `other_source_urls` | text | extra pages, joined with a vertical bar |
| `quote` | text | the place name as printed on the list, for supported rows, otherwise empty |
| `note` | text | place-specific remark |
| `product_note` | text | remark that applies to the whole product |
| `epoch_organization` | text | matching `organization_primary` value in notebook 01, for the join |

### 2.2 Coding scheme

- `supported`: the place appears in the provider's official supported list for that product.
- `not_listed`: the provider publishes a list and the place is absent. This means "not on the list", not proof of a technical block.
- `excluded_stated`: a page says in words that the place is excluded or that service is not offered there. No row has this value in this version.
- `no_list_found`: no official country list was found. Availability is unknown and not assumed.
- Scope: API or developer platform only. Consumer chat apps often follow different rules.

## 3. Set up

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

T0 = time.time()
FOLDER = "03_search_c_provider_availability"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\03_search_c_provider_availability
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw


## 4. The records

Each provider is typed once with its four place codes. The notebook then expands them into one row per place.

In [2]:
VIEWED = "2026-10-16"
PLACES = ["Hong Kong", "Mainland China", "Singapore", "Taiwan"]

PRODUCTS = [
    dict(provider="OpenAI", product="API", epoch_organization="OpenAI", page_date=None,
         status=dict(zip(PLACES, ["not_listed", "not_listed", "supported", "supported"])),
         source_url="https://developers.openai.com/api/docs/supported-countries", other="",
         product_note="Macau is also absent. The old platform.openai.com address redirects (HTTP 302) to the new page."),
    dict(provider="Anthropic", product="API and Claude", epoch_organization="Anthropic", page_date=None,
         status=dict(zip(PLACES, ["not_listed", "not_listed", "supported", "supported"])),
         source_url="https://www.anthropic.com/supported-countries", other="",
         product_note="Macau is also absent. Ukraine is listed with named regions excluded. No last-updated line, footer copyright 2026."),
    dict(provider="Google", product="Gemini API", epoch_organization="Google DeepMind", page_date="2026-04-28",
         status=dict(zip(PLACES, ["not_listed", "not_listed", "supported", "supported"])),
         source_url="https://ai.google.dev/gemini-api/docs/available-regions", other="",
         product_note="The page points unlisted regions to a Google enterprise cloud route. We did not open that route and make no claim about whether it works in Hong Kong."),
    dict(provider="DeepSeek", product="Open Platform API", epoch_organization="DeepSeek", page_date="2026-04-29",
         status=dict(zip(PLACES, ["no_list_found"] * 4)),
         source_url="https://cdn.deepseek.com/policies/en-UK/deepseek-open-platform-terms-of-service.html",
         other="https://api-docs.deepseek.com/",
         product_note="Section 1.4 gives no warranty that the service is available in certain jurisdictions. The sanctions clause names no places."),
    dict(provider="Alibaba Cloud", product="Model Studio", epoch_organization="Alibaba", page_date=None,
         status=dict(zip(PLACES, ["no_list_found"] * 4)),
         source_url="https://www.alibabacloud.com/help/en/model-studio/regions.md", other="",
         product_note="Offers a China (Hong Kong) deployment region and a Singapore region. The FAQ says all models can be called from mainland China. Each region has its own endpoint and API key. A deployment region is not a country list."),
    dict(provider="Z.ai", product="API (international)", epoch_organization="Z.ai (Zhipu AI)", page_date="2026-04-14",
         status=dict(zip(PLACES, ["no_list_found"] * 4)),
         source_url="https://docs.z.ai/legal-agreement/terms-of-use.md",
         other="https://docs.z.ai/guides/overview/overview",
         product_note="Terms exclude Iran, North Korea, Cuba, Crimea, Donetsk and Zaporizhzhia. Hong Kong and mainland China are not named."),
    dict(provider="Moonshot AI (Kimi)", product="Open platform (international)", epoch_organization="Moonshot", page_date="2026-07-30",
         status=dict(zip(PLACES, ["no_list_found"] * 4)),
         source_url="https://platform.kimi.ai/docs/agreement/modeluse", other="",
         product_note="Section 13 bars use in countries under comprehensive sanctions. No place is named."),
]

PLACE_NOTE = {
    "supported": "place appears on the supported list",
    "not_listed": "place absent from the published list",
    "no_list_found": "no official country list found, availability unknown",
    "excluded_stated": "page states the place is excluded",
}

rows = []
for p in PRODUCTS:
    for place in PLACES:
        status = p["status"][place]
        rows.append(dict(
            provider=p["provider"], product=p["product"], site="international", place=place, status=status,
            page_date=p["page_date"], date_viewed=VIEWED, source_url=p["source_url"],
            other_source_urls=p["other"] or None,
            quote=place if status == "supported" else None,
            note=PLACE_NOTE[status], product_note=p["product_note"], epoch_organization=p["epoch_organization"]))
print(len(rows), "rows built from", len(PRODUCTS), "products")

28 rows built from 7 products


## 5. Build the table and validate it

In [3]:
ALLOWED_STATUS = {"supported", "not_listed", "excluded_stated", "no_list_found"}
table = pd.DataFrame(rows)
table["page_date"] = pd.to_datetime(table["page_date"], errors="coerce")
table["date_viewed"] = pd.to_datetime(table["date_viewed"])

problems = []
bad = table.loc[~table["status"].isin(ALLOWED_STATUS), "status"]
if len(bad):
    problems.append(f"status outside the list: {sorted(bad.unique())}")
if table.duplicated(subset=["provider", "product", "place"]).any():
    problems.append("duplicated provider, product and place")
if len(table) != len(PRODUCTS) * len(PLACES):
    problems.append("row count is not products times places")
if not table["source_url"].str.startswith("http").all():
    problems.append("a row has no valid source_url")
# A provider that publishes a list must have both supported and not_listed or only supported. A no_list_found product must be all no_list_found.
for (prov, prod), g in table.groupby(["provider", "product"]):
    kinds = set(g["status"])
    if "no_list_found" in kinds and kinds != {"no_list_found"}:
        problems.append(f"{prov} {prod} mixes no_list_found with other values")
# The one claim in the Search C summary, tested against the table:
listed = table[table["provider"].isin(["OpenAI", "Anthropic", "Google"])]
hk_cn = listed[listed["place"].isin(["Hong Kong", "Mainland China"])]
sg_tw = listed[listed["place"].isin(["Singapore", "Taiwan"])]
if not hk_cn["status"].eq("not_listed").all() or not sg_tw["status"].eq("supported").all():
    problems.append("the summary claim (HK and China absent, Singapore and Taiwan listed for OpenAI, Anthropic, Google) does not hold")

epoch_file = DATA / "01_epoch_eci_scores" / "processed" / "model_capability.csv"
if epoch_file.exists():
    epoch_orgs = set(pd.read_csv(epoch_file)["organization_primary"].dropna())
    lost = sorted(set(table["epoch_organization"]) - epoch_orgs)
    print("Organisation names that Epoch does not use:", lost if lost else "none")
    if lost:
        problems.append(f"join would drop {lost}")
else:
    print("Notebook 01 has not been run yet, so the join test was skipped.")

print("Problems:", problems if problems else "none")
assert not problems, problems
table.head(4)

Organisation names that Epoch does not use: none
Problems: none


,provider,product,site,place,status,page_date,date_viewed,source_url,other_source_urls,quote,note,product_note,epoch_organization
0,OpenAI,API,international,Hong Kong,not_listed,NaT,2026-10-16,https://developers.openai.com/api/docs/support...,None,None,place absent from the published list,Macau is also absent. The old platform.openai....,OpenAI
1,OpenAI,API,international,Mainland China,not_listed,NaT,2026-10-16,https://developers.openai.com/api/docs/support...,None,None,place absent from the published list,Macau is also absent. The old platform.openai....,OpenAI
2,OpenAI,API,international,Singapore,supported,NaT,2026-10-16,https://developers.openai.com/api/docs/support...,None,Singapore,place appears on the supported list,Macau is also absent. The old platform.openai....,OpenAI
3,OpenAI,API,international,Taiwan,supported,NaT,2026-10-16,https://developers.openai.com/api/docs/support...,None,Taiwan,place appears on the supported list,Macau is also absent. The old platform.openai....,OpenAI


## 6. Optional: check that the source links still open

Off by default. It sends about 10 requests and some sites block automated requests even when the page works in a browser, so a failure is a prompt to open the page by hand. Set `CHECK_LINKS = True` to run it. Expect 10 to 30 seconds.

In [4]:
CHECK_LINKS = False

if CHECK_LINKS:
    import requests
    urls = sorted(set(table["source_url"]) | {u for cell in table["other_source_urls"].dropna() for u in cell.split("|")})
    results = []
    for url in urls:
        try:
            status = requests.get(url, headers={"User-Agent": "UTS-36104-A3-student-project (academic use)"},
                                  timeout=20, allow_redirects=True).status_code
        except Exception as err:
            status = type(err).__name__
        results.append({"url": url, "status": status})
        time.sleep(0.5)
    display(pd.DataFrame(results))
else:
    print("Link check skipped (CHECK_LINKS is False).")

Link check skipped (CHECK_LINKS is False).


## 7. Save

In [5]:
table.to_csv(OUT / "provider_availability.csv", index=False, encoding="utf-8")
print("saved", OUT / "provider_availability.csv", table.shape)

saved d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\03_search_c_provider_availability\processed\provider_availability.csv (28, 13)


## 8. Quick look: provider by place

In [6]:
table.pivot(index=["provider", "product"], columns="place", values="status")[PLACES]

,place,Hong Kong,Mainland China,Singapore,Taiwan
provider,product,,,,
Alibaba Cloud,Model Studio,no_list_found,no_list_found,no_list_found,no_list_found
Anthropic,API and Claude,not_listed,not_listed,supported,supported
DeepSeek,Open Platform API,no_list_found,no_list_found,no_list_found,no_list_found
Google,Gemini API,not_listed,not_listed,supported,supported
Moonshot AI (Kimi),Open platform (international),no_list_found,no_list_found,no_list_found,no_list_found
OpenAI,API,not_listed,not_listed,supported,supported
Z.ai,API (international),no_list_found,no_list_found,no_list_found,no_list_found


## How to update

There is no file to download. To bring the table up to date:

1. Open the seven links in Section 1. For OpenAI, Anthropic and Google, check whether Hong Kong, Mainland China, Singapore and Taiwan are on the published list. For the other four providers, check again whether an official country list now exists.
2. In the cell `PRODUCTS`, change the four status codes of a provider where the page changed. The allowed values are in Section 2.2. Change `page_date` to the date printed on the page, or `None` if the page shows none.
3. Change `VIEWED` at the top of the same cell to the day you read the pages.
4. Run all cells. Section 5 stops with a message if a status is outside the list, a row is duplicated, or the summary claim (Hong Kong and China absent, Singapore and Taiwan listed for OpenAI, Anthropic and Google) no longer holds. If the claim no longer holds, that is a finding to report and not a bug to hide.
5. Optional: set `CHECK_LINKS = True` in Section 6 to test the links.

Reminder: "not on the list" is not proof of a block. Do not use this notebook for advice on getting around any restriction.

**Update check.** Open `processed/data_currency.csv`. `data_covers_to` should be your new viewing day.

## 9. Run time and limits

**Time to extract.** Nothing is downloaded, so the run takes **under 2 seconds**. The optional link check adds 10 to 30 seconds.

**Limits and known risks.**

1. "Not on the list" is not the same as "blocked". The pages say where service is offered and not how it is enforced.
2. Page text came through a page reader that returns extracts. Check each list by eye before quoting it.
3. OpenAI and Anthropic pages show no date, so the only date is our viewing date.
4. The Google page mentions an enterprise route for unlisted regions that we did not open.
5. China-site products are not covered here. Zhipu bigmodel.cn and Kimi platform.kimi.com are separate China services and we did not check who can sign up.
6. Cloud resellers are not covered. Whether the same models are reachable through a cloud provider's Hong Kong or Singapore region is a separate search. Notebook 04 (LiteLLM) shows which hosts list each model.
7. MiniMax is not in this table.
8. Several of these pages changed during 2026. Re-check each one before submission.

## 10. References

1. Project note: `04 Searches A B C/Search C Provider Availability (English).md`
2. OpenAI, Supported countries and territories. https://developers.openai.com/api/docs/supported-countries
3. Anthropic, Supported countries and regions. https://www.anthropic.com/supported-countries
4. Google, Gemini API available regions. https://ai.google.dev/gemini-api/docs/available-regions
5. DeepSeek, Open Platform Terms of Service. https://cdn.deepseek.com/policies/en-UK/deepseek-open-platform-terms-of-service.html
6. DeepSeek, API docs home. https://api-docs.deepseek.com/
7. Alibaba Cloud, Model Studio regions. https://www.alibabacloud.com/help/en/model-studio/regions.md
8. Z.ai, Terms of Use. https://docs.z.ai/legal-agreement/terms-of-use.md
9. Z.ai, docs overview. https://docs.z.ai/guides/overview/overview
10. Kimi open platform, Terms. https://platform.kimi.ai/docs/agreement/modeluse

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [7]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


pg = table["page_date"].dropna()
write_currency("Search C provider availability (provider_availability.csv)",
               pg.min() if len(pg) else None, table["date_viewed"].max(),
               "hand-built table, pages read on 2026-10-16. Oldest page date is shown. OpenAI, Anthropic and Alibaba pages show no date",
               "Each page's own last-updated date is in page_date where shown",
               "None. Re-check before submission, several pages changed during 2026")

dataset                  Search C provider availability (provider_availability.csv)
extracted_utc            2026-10-17T22:27:51Z
data_covers_from         2026-04-14
data_covers_to           2026-10-16
snapshot_note            hand-built table, pages read on 2026-10-16. Oldest page date is shown. OpenAI, Anthropic and Alibaba pages show no date
source_origin_note       Each page's own last-updated date is in page_date where shown
source_update_frequency  None. Re-check before submission, several pages changed during 2026


In [8]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 0.5 seconds
Files written:
  processed\data_currency.csv  (469 bytes)
  processed\provider_availability.csv  (9,673 bytes)
